# 00B — Import existing cells/bins without resegmentation

Convenience alternative to notebooks 01/02/05 for existing datasets. It only runs import/reuse actions.
It does **not** call StarDist, Proseg, fresh H&E QC, assembly or publication. Review the imported coordinate plots before notebook 06.
For `cell_annotated + export_only`, use notebook 12 instead.

In [ ]:
from pathlib import Path
import os, sys, json
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "vhd" / "control").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Start Jupyter in the extracted pipeline folder (or its notebooks folder).")
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from vhd.control.manifest import load_project, check_policy, save_plan, sample_layout
MANIFEST = Path(os.environ.get("VHD_MANIFEST", ROOT / "config" / "samples.csv"))
SETTINGS = Path(os.environ.get("VHD_SETTINGS", ROOT / "config" / "settings.json"))
if not MANIFEST.exists() or not SETTINGS.exists():
    raise FileNotFoundError("Copy a supplied samples.*.csv to config/samples.csv and settings.g5_24xlarge.example.json to config/settings.json; edit paths and policy first.")
PROJECT = load_project(MANIFEST, SETTINGS)
check_policy(PROJECT)
# Default is a dry run. Set True here only after reviewing the printed plan.
EXECUTE = os.environ.get("VHD_EXECUTE", "0") == "1"
# Optional pilot selection, e.g. ["StudyLegacy__Sample01"]. None selects all applicable rows.
SAMPLE_KEYS = None


## Select applicable migration rows; set SAMPLE_KEYS for a one-sample pilot

In [ ]:
from vhd.compute.launch import launch_samples
keys = [r["sample_key"] for r in PROJECT["rows"]
        if r["stage"] in ("bin_QCed", "cell_segmented", "cell_annotated")
        and r["goal"] != "export_only"
        and (SAMPLE_KEYS is None or r["sample_key"] in SAMPLE_KEYS)]
launch_samples(PROJECT, ["import_bins", "reuse_bin_qc", "import_cells"], execute=EXECUTE, sample_keys=keys)